# CaseGuide: traffic tickets, court records, and a toy classifier

Through Insight Data Science I worked on: **CaseGuide**. I ended up developing a tool that could take messy traffic-court data and help people get a *rough* read on outcomes.

Working with CaseGuide, I got my hands on civil-traffic records from a **Florida county** through a **Freedom of Information Act** request. The questions I cared about were pretty simple: when someone actually **fights** a speeding ticket instead of paying the civil penalty, what tends to happen? How does that relate to whether they had **defense counsel**, how far over the limit they were going, and the **demographic fields** the court files happened to include?

This notebook is me walking through the same pipeline I used for the analysis that fed a slide deck and a prototype tool: **clean and anonymize** the extract, **engineer features**, deal with **class imbalance**, fit a **random forest** (plus calibration) and an **XGBoost** model, and poke at **feature importances** and a couple of charts. It’s written in the same spirit as my other writeups—here’s what I did, here’s where I’d be careful—rather than a polished product spec.

Below are a few figures straight from my **Week 4 Insight Data Science** deck so the notebook isn’t *only* code; then we load libraries and get into the weeds.



## Slides from the Week 4 deck

*Slide 1 — title.*

![CaseGuide / Traffic Tool title slide](images/caseguide/slide_01.png)

*Slide 2 — the “Kelley Blue Book for court cases” idea: predicted outcomes and (in the full product vision) attorney suggestions for criminal traffic.*

![Kelly Blue Book for court cases concept slide](images/caseguide/slide_02.png)

*Slide 3 — why traffic data is worth caring about: cost of tickets, insurance, court fees, and what CaseGuide was trying to surface for users.*

![Traffic data and user engagement slide](images/caseguide/slide_03.png)


*Slide 4 — before a court IT overhaul, most records were pay-without-contest; only a small slice were “fought” cases—the slice this project focuses on.*

![Case mix before court overhaul](images/caseguide/slide_04.png)

*Slide 5 — millions of raw records collapse to tens of thousands of cases after cleaning; random forest + calibrated probabilities on anonymized features.*

![Pipeline: raw files to classifier](images/caseguide/slide_05.png)

*Slide 6 — outcome labels (1 not guilty / 2 withheld / 3 guilty) and the modeling stack.*

![Outcomes and model stack](images/caseguide/slide_06.png)


## The questions I actually wanted to answer

- Among closed civil-traffic cases with usable demographic fields, what **outcomes** show up (guilty vs withhold / diversion vs dismissed or dismissed-like)?
- Do simple things we can observe—**race**, **gender**, **age**, **whether there was defense counsel**, **priors**, **speed over the limit**—help predict **outcome** on a held-out split? (Not “prove” anything about fairness—just quantify what’s in the rows.)
- After **undersampling** and **calibration**, how useful are the **probabilities**, and which **features** does the forest lean on?

I’m treating the models as **descriptive / predictive on past data**, not causal. If you want to skip ahead to the code, jump to **Setup** below.


## Setup

I'll use **pandas** for tabular work, **scikit-learn** for the random forest, calibration, and metrics, **imbalanced-learn** for undersampling, **matplotlib** for a validation curve, and **Plotly** for an interactive stacked bar (inline in Jupyter—no Plotly Cloud account needed).


In [ ]:
import calendar

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.under_sampling import RandomUnderSampler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, log_loss
from sklearn.model_selection import train_test_split, validation_curve
from xgboost import XGBClassifier

import plotly.graph_objects as go

pd.set_option("display.max_columns", 500)

sns.set_theme(style="whitegrid")


## 1. Clean and restrict cases

Raw extracts include many columns we do not need for this analysis. We keep **closed** civil-traffic cases, drop impossible speeds, require a **judge**, restrict to offenses after 2014, and require **gender** and **race** (non-null) for the fairness-related analyses we planned.

We also drop rows that clearly represent **pay without contest**-style resolutions (no real contest), and restrict to rows with a **plea** recorded so “fighting the ticket” is meaningful in the data we model.


In [ ]:
DROP_IRRELEVANT = [
    "CaseType", "AlcoholLevel1", "AlcoholLevel2", "AlcoholTestRefused", "ConfinementType",
    "MaxConfinement", "CommunityControl", "Probation", "CommunityService", "CreditTimeServed",
]


def clean_data(cg: pd.DataFrame) -> pd.DataFrame:
    cg = cg.drop(columns=DROP_IRRELEVANT, errors="ignore")
    cg = cg.loc[cg["CaseStatus"] == "Closed"].copy()
    cg = cg.loc[cg["ActualSpeed"] < 200]
    cg = cg.dropna(subset=["Judge"])
    cg = cg.loc[cg["OffenseDate"] > "2014-01-01"]
    cg = cg.dropna(subset=["Gender", "Race"])

    mask_no_fight = (cg["PleaDescription"] == "No Plea Entered") & (
        cg["DispositionDescription"] == "PAY CIVIL PENALTY GUILTY"
    )
    cg = cg.loc[~mask_no_fight]
    mask_hearing = cg["Judge"].isin(["98 HEARING  OFFICER", "95 HEARING  OFFICER"]) & (
        cg["PleaDescription"] == "No Plea Entered"
    )
    cg = cg.loc[~mask_hearing]
    cg = cg.dropna(subset=["PleaDescription"])
    return cg


## 2. Features, outcome coding, and anonymization

We derive a few interpretable fields:

- **ZipCode** (last five digits of address) as a coarse geography proxy  
- **SpeedDiff** (actual minus posted speed)  
- **nthOccurrence** (which numbered case this is for the same defendant id in the extract)  
- **Ageatfile** from file date minus date of birth  
- **HasDefense** (whether a defense attorney field suggests counsel)  
- **WasArrested**, **FloridaDL**, month of offense, and cleaned **Gender** / **AccidentIndicator** labels  

**Disposition** is mapped to three ordered buckets for multiclass modeling: **1** = favorable to defendant (dismissed, nolle, acquittal, etc.), **2** = withhold / diversion / transfer-like, **3** = guilty. (Exact legal labels vary; this coding matches the original project.)

To limit sparsity, we keep only the **top-K** statutes, **top-K** defense-attorney labels, and **top-K** ZIPs by frequency. Finally we **drop direct identifiers** (UCN, names, DL numbers, etc.) before modeling.


In [ ]:
TOP_STATUTE = 74
TOP_ATTORNEY = 70
TOP_ZIP = 79


def get_new_features(cg: pd.DataFrame) -> pd.DataFrame:
    cg = cg.copy()
    cg["ZipCode"] = cg["Address"].str[-5:]
    cg["SpeedDiff"] = cg["ActualSpeed"] - cg["PostedSpeed"]
    cg["nthOccurrence"] = cg.groupby("Defendant").cumcount() + 1

    cg["Ageatfile"] = (cg["FileDate"] - cg["DOB"]).dt.days / 365.24
    cg = cg.loc[cg["Ageatfile"] < 100]

    attorney = cg["DefendantLeadAttorney"].fillna("")
    cg["HasDefense"] = np.where(attorney.str.contains("A", regex=False), "Yes", "No")

    cg["WasArrested"] = np.where(~cg["ArrestDate"].isna(), "Yes", "No")
    cg["FloridaDL"] = np.where(cg["DLState"].str.contains("Florida", na=False), "Yes", "No")

    cg["OffenseMonth"] = cg["OffenseDate"].dt.month.map(lambda m: calendar.month_abbr[m])

    cg["AccidentIndicator"] = cg["AccidentIndicator"].str.replace("Y", "Yes").str.replace("N", "No")
    cg["Gender"] = cg["Gender"].str.replace("M", "Male").str.replace("F", "Female")

    # Multiclass label (1 = favorable, 2 = withhold/diversion, 3 = guilty). Order matches original project.
    dd = cg["DispositionDescription"].fillna("")
    cg["Disposition"] = 3
    cg.loc[dd.str.contains("GUILTY", case=False, na=False), "Disposition"] = 3
    cg.loc[dd.str.contains("WITHHOLD", case=False, na=False), "Disposition"] = 2
    cg.loc[dd.str.contains("WITHHELD", case=False, na=False), "Disposition"] = 2
    cg.loc[dd.str.contains("A/WH", case=False, na=False), "Disposition"] = 2
    cg.loc[dd.str.contains("TRANSFER", case=False, na=False), "Disposition"] = 2
    cg.loc[dd.str.contains("DIVERSION", case=False, na=False), "Disposition"] = 2
    cg.loc[dd.str.contains("DISMISSED", case=False, na=False), "Disposition"] = 1
    cg.loc[dd.str.contains("NOLLE", case=False, na=False), "Disposition"] = 1
    cg.loc[dd.str.contains("DECLINE", case=False, na=False), "Disposition"] = 1
    cg.loc[dd.str.contains("NOT", case=False, na=False), "Disposition"] = 1
    cg.loc[dd.str.contains("ACQUITTED", case=False, na=False), "Disposition"] = 1
    cg.loc[dd.str.contains("NO ACTION TAKEN", case=False, na=False), "Disposition"] = 1

    top_statute = cg["Statute"].value_counts().head(TOP_STATUTE).index
    top_atty = cg["DefendantLeadAttorney"].value_counts().head(TOP_ATTORNEY).index
    top_zip = cg["ZipCode"].value_counts().head(TOP_ZIP).index
    cg = cg.loc[cg["Statute"].isin(top_statute) & cg["DefendantLeadAttorney"].isin(top_atty) & cg["ZipCode"].isin(top_zip)]

    drop_privacy = [
        "UCN", "CaseStyle", "CaseNumber", "CaseStatus", "CitationNumber", "VehicleLicense",
        "Defendant", "DOB", "Address", "DLNum",
    ]
    drop_extra = [
        "ActualSpeed", "PostedSpeed", "ChargeOffenseDescription", "CaseStatusDate", "FileDate",
        "OffenseDate", "DLState", "ArrestDate", "PleaEventDate", "Event_Date", "DispositionDescription",
    ]
    cg = cg.drop(columns=drop_privacy + drop_extra, errors="ignore")
    return cg


## 3. Load cleaned modeling table

The project saved a pickled dataframe as `civil_traffic.pkl` next to this notebook (not committed to git if large). If you do not have the file, you cannot run the cells below; the code is still readable as documentation.


In [ ]:
cases = pd.read_pickle("civil_traffic.pkl")
cases = clean_data(cases)
cases = get_new_features(cases)
cases.head()


## 4. Encode categoricals for sklearn

We drop high-cardinality **free-text** fields used only for other views (officer names, vehicle make, judge name, plea text, attorney name in raw form—we already derived **HasDefense**). Remaining categoricals are **one-hot encoded**. Numeric fields **SpeedDiff** and **Ageatfile** have missing values filled with the **training** mean inside `prep_for_model` (in a production pipeline you would use an imputer fit on train only).


In [ ]:
CAT_COLS = [
    "Race", "Gender", "AccidentIndicator", "HasDefense", "WasArrested",
    "FloridaDL", "Statute", "ZipCode", "OffenseMonth",
]


def prep_for_model(cg: pd.DataFrame) -> pd.DataFrame:
    drop_cols = [
        "ArrestOfficer", "ArrestAgency", "CitationAgency", "CitationOfficer",
        "VehicleMakeModel", "Judge", "PleaDescription", "DefendantLeadAttorney",
    ]
    cg_m = cg.drop(columns=drop_cols, errors="ignore")
    cg_m = pd.get_dummies(cg_m, columns=CAT_COLS, prefix=CAT_COLS, prefix_sep="_", dtype=int)
    cg_m["SpeedDiff"] = cg_m["SpeedDiff"].fillna(cg_m["SpeedDiff"].mean())
    cg_m["Ageatfile"] = cg_m["Ageatfile"].fillna(cg_m["Ageatfile"].mean())
    return cg_m


model_data = prep_for_model(cases)


## 5. Train / validation / test split and class imbalance

We hold out **20%** for a final test set, then take **25%** of the remainder as validation. **RandomUnderSampler** balances the training fold so the forest is not overwhelmed by the majority disposition—at the cost of discarding majority examples (a tradeoff appropriate for exploration; for production you might prefer class weights or other strategies).


In [ ]:
rf = RandomForestClassifier(n_estimators=25, max_depth=50, oob_score=True, random_state=4, n_jobs=-1)

X = model_data.drop(columns=["Disposition"])
y = model_data["Disposition"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4, stratify=y)
X_traintrain, X_validate, y_traintrain, y_validate = train_test_split(
    X_train, y_train, test_size=0.25, random_state=4, stratify=y_train
)

rus = RandomUnderSampler(random_state=0, replacement=False, sampling_strategy="not minority")
X_traintrain, y_traintrain = rus.fit_resample(X_traintrain, y_traintrain)


## 6. Random forest: fit and multiclass F1

We report **per-class F1** on the validation set (classes 1, 2, 3 as coded above). **OOB score** is available from the forest if you inspect `rf.oob_score_` after fitting.


In [ ]:
rf.fit(X_traintrain, y_traintrain)
val_preds = rf.predict(X_validate)
f1_score(y_validate, val_preds, average=None)


## 7. Probability calibration and log loss

Tree ensembles can be **miscalibrated**. We wrap the already-fit forest with **sigmoid calibration** (`CalibratedClassifierCV`, `prefit`) using the full training split, then evaluate **log loss** on the held-out test probabilities—a stricter check than accuracy when probabilities matter.


In [ ]:
sig_clf = CalibratedClassifierCV(rf, method="sigmoid", cv="prefit")
sig_clf.fit(X_train, y_train)
test_probs = sig_clf.predict_proba(X_test)
log_loss(y_test, test_probs)


## 8. Feature importances (Gini)

A quick view of which engineered columns the forest relied on most (importances sum to 1 across features).


In [ ]:
feat_cols = getattr(X_traintrain, "columns", None) or X_train.columns
importances = pd.DataFrame({"feature": feat_cols, "importance": rf.feature_importances_})
importances.sort_values("importance", ascending=False).head(20)


## 9. Exploratory chart: outcomes by defense counsel (Plotly)

The stacked bar shows **row percentages** of disposition bucket within **HasDefense** Yes vs No. This mirrors the kind of view the CaseGuide prototype aimed to support: **broad** comparisons, not individual legal advice.

**Note:** `cases` is the cleaned dataframe; we avoid reusing the name `data` so we do not overwrite it with Plotly traces.


In [ ]:
pct = (
    cases.groupby(["HasDefense", "Disposition"])
    .size()
    .unstack(fill_value=0)
    .pipe(lambda t: t.div(t.sum(axis=1), axis=0) * 100)
)
pct = pct.reindex(columns=[1, 2, 3], fill_value=0)
x_vals = pct.index.tolist()
traces = [
    go.Bar(name="Not guilty / dismissed", x=x_vals, y=pct[1]),
    go.Bar(name="Withheld or transferred", x=x_vals, y=pct[2]),
    go.Bar(name="Guilty", x=x_vals, y=pct[3]),
]
fig = go.Figure(data=traces, layout=go.Layout(
    barmode="stack",
    title="Case outcomes by whether defense counsel appears in the record",
    xaxis_title="Recorded defense counsel (A in attorney field)",
    yaxis_title="Percent within Yes/No",
    plot_bgcolor="rgba(230,230,230,0.9)",
))
fig.show()


## 10. XGBoost and validation curve

We fit a **multiclass XGBoost** classifier on the same undersampled training matrix and report validation F1. The **validation curve** varies `n_estimators` to illustrate bias–variance tradeoffs; it uses **3-fold CV** on the undersampled training data.

**Fix vs original notebook:** the step size in `range(10, 200, …)` must be valid Python (`10`, not a typo). The plot title references **XGBoost**, not random forest.


In [ ]:
xgb = XGBClassifier(
    learning_rate=0.1,
    n_estimators=100,
    max_depth=3,
    objective="multi:softprob",
    num_class=3,
    random_state=0,
    n_jobs=-1,
)
xgb.fit(X_traintrain, y_traintrain)
xgb_pred = xgb.predict(np.asarray(X_validate))
f1_score(y_validate, xgb_pred, average=None)


In [ ]:
param_range = range(10, 200, 10)
train_scores, test_scores = validation_curve(
    XGBClassifier(
        learning_rate=0.1,
        max_depth=3,
        objective="multi:softprob",
        num_class=3,
        random_state=0,
        n_jobs=-1,
    ),
    X_traintrain,
    y_traintrain,
    param_name="n_estimators",
    param_range=param_range,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.plot(param_range, train_mean, label="Training score", color="black")
plt.plot(param_range, test_mean, label="Cross-validation score", color="dimgray")
plt.fill_between(param_range, train_mean - train_std, train_mean + train_std, color="gray", alpha=0.3)
plt.fill_between(param_range, test_mean - test_std, test_mean + test_std, color="gainsboro", alpha=0.5)
plt.title("Validation curve: XGBoost (n_estimators)")
plt.xlabel("Number of trees")
plt.ylabel("Accuracy")
plt.tight_layout()
plt.legend(loc="best")
plt.show()


## Limitations

- **One jurisdiction, one extract**—patterns may not transfer.  
- **Residual confounding**—unobserved factors drive many outcomes.  
- **Demographics** are recorded as in court data; interpretation belongs to subject-matter experts.  
- Models are **retrospective**; they do not predict any individual’s case.

For a slide-style overview of motivation and design, see the Week 4 deck that accompanied this work.
